# 🌬️ AirQI — Air Quality Analysis & AQI Prediction
**Author:** Samikshya Saud  
**Project:** Air Quality Analysis using Machine Learning  
**Dataset:** US EPA Air Quality Data (Sample)

---
### What this notebook does:
1. Load and explore air quality dataset
2. Clean and preprocess the data
3. Visualize pollution trends
4. Build ML models to predict AQI
5. Evaluate model performance

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Step 2: Load and Explore Dataset

In [ ]:
# Load dataset
df = pd.read_csv('../data/us_air_quality_sample.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Basic information about the dataset
print('Dataset Info:')
print(df.info())
print('\nBasic Statistics:')
df.describe()

In [ ]:
# Check for missing values
print('Missing Values in Each Column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

## Step 3: Data Cleaning & Preprocessing

In [ ]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract month for seasonal analysis
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%B')

print('Date conversion successful!')
print('Unique cities:', df['city'].unique())
print('Unique AQI categories:', df['category'].unique())

In [ ]:
# Check AQI category distribution
print('AQI Category Distribution:')
print(df['category'].value_counts())
print(f'\nPercentage of Good/Moderate days: {round(len(df[df["AQI"] <= 100]) / len(df) * 100, 1)}%')

## Step 4: Data Visualization

In [ ]:
# Plot 1: AQI Distribution
plt.figure(figsize=(10, 5))
plt.hist(df['AQI'], bins=20, color='steelblue', edgecolor='white')
plt.title('Distribution of Air Quality Index (AQI)', fontsize=14)
plt.xlabel('AQI Value')
plt.ylabel('Frequency')
plt.axvline(x=50, color='green', linestyle='--', label='Good (50)')
plt.axvline(x=100, color='orange', linestyle='--', label='Moderate (100)')
plt.axvline(x=150, color='red', linestyle='--', label='Unhealthy (150)')
plt.legend()
plt.tight_layout()
plt.savefig('aqi_distribution.png', dpi=100)
plt.show()
print('Plot saved!')

In [ ]:
# Plot 2: PM2.5 Trend Over Time by City
plt.figure(figsize=(12, 5))
for city in df['city'].unique():
    city_data = df[df['city'] == city].sort_values('date')
    plt.plot(city_data['date'], city_data['PM2_5'], marker='o', label=city, linewidth=2)

plt.title('PM2.5 Pollution Trend Over Time', fontsize=14)
plt.xlabel('Date')
plt.ylabel('PM2.5 (μg/m³)')
plt.axhline(y=35, color='red', linestyle='--', label='WHO Limit (35)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('pm25_trend.png', dpi=100)
plt.show()
print('Plot saved!')

In [ ]:
# Plot 3: Average AQI by City
plt.figure(figsize=(8, 5))
city_avg = df.groupby('city')['AQI'].mean().sort_values(ascending=False)
bars = plt.bar(city_avg.index, city_avg.values, color=['tomato', 'steelblue'])
plt.title('Average AQI by City', fontsize=14)
plt.xlabel('City')
plt.ylabel('Average AQI')
for bar, val in zip(bars, city_avg.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('avg_aqi_city.png', dpi=100)
plt.show()

In [ ]:
# Plot 4: Correlation Heatmap of Pollutants
plt.figure(figsize=(9, 7))
pollutants = ['PM2_5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'AQI']
corr_matrix = df[pollutants].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.5, square=True)
plt.title('Correlation Between Pollutants and AQI', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100)
plt.show()
print('Correlation heatmap saved!')

## Step 5: Feature Engineering & ML Model

In [ ]:
# Prepare features and target
features = ['PM2_5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'month']
target = 'AQI'

X = df[features]
y = df[target]

# Split into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')
print(f'Features used:    {features}')

In [ ]:
# Train 3 models and compare
models = {
    'Linear Regression':        LinearRegression(),
    'Random Forest':            RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':        GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    results.append({'Model': name, 'MAE': round(mae, 3),
                    'MSE': round(mse, 3), 'R2 Score': round(r2, 3)})
    print(f'{name}:')
    print(f'  MAE:      {mae:.3f}')
    print(f'  MSE:      {mse:.3f}')
    print(f'  R² Score: {r2:.3f}')
    print()

In [ ]:
# Compare model performance
results_df = pd.DataFrame(results)
print('Model Comparison:')
print(results_df)

best_model = results_df.loc[results_df['R2 Score'].idxmax(), 'Model']
best_r2    = results_df['R2 Score'].max()
print(f'\nBest Model: {best_model} with R² Score = {best_r2}')

In [ ]:
# Plot: Feature Importance from Random Forest
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=features)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importance — Random Forest', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100)
plt.show()
print(f'Most important pollutant for AQI prediction: {importances.idxmax()}')

## Step 6: Key Findings & Conclusions

In [ ]:
# Summary statistics
print('='*50)
print('KEY FINDINGS')
print('='*50)
print(f'Total records analyzed:     {len(df)}')
print(f'Cities covered:             {df["city"].nunique()}')
print(f'Date range:                 {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Average AQI:                {df["AQI"].mean():.1f}')
print(f'Highest AQI recorded:       {df["AQI"].max()} ({df.loc[df["AQI"].idxmax(), "city"]})')
print(f'Good air quality days:      {len(df[df["AQI"] <= 50])} ({round(len(df[df["AQI"] <= 50])/len(df)*100, 1)}%)')
print(f'Unhealthy air quality days: {len(df[df["AQI"] > 100])} ({round(len(df[df["AQI"] > 100])/len(df)*100, 1)}%)')
print(f'Best ML Model:              {best_model} (R² = {best_r2})')
print(f'Most important pollutant:   {importances.idxmax()}')